# ex05 · LeNet（对应教材 6.6）

> **做题流程**：组装 LeNet 并训练 Fashion-MNIST，观察每层尺寸。
> **做完再看** `solutions/ex05-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> LeNet = 卷积 → 池化 → 卷积 → 池化 → 全连接。

## 题 1 🔧 组装 LeNet（TODO 6.7）

补全 make_lenet。结构：两个「卷积+ReLU+池化」块，再接 Flatten 和三个全连接层（尺寸链见下方运行结果）。

In [5]:
import torch
from torch import nn
import torchvision
from torch.utils import data
from torchvision import transforms

def make_lenet():
    # TODO 6.7: 返回 nn.Sequential 的 LeNet
    # Conv2d(1,6,5,padding=2)+ReLU+MaxPool(2,2) → Conv2d(6,16,5)+ReLU+MaxPool(2,2)
    # → Flatten → Linear(16*5*5,120)+ReLU → Linear(120,84)+ReLU → Linear(84,10)
    lenet = nn.Sequential(nn.Conv2d(1, 6, 5, padding = 2), # n, 6, 28, 28
                          nn.ReLU(),
                          nn.MaxPool2d(2, 2), # n, 6, 14, 14

                          nn.Conv2d(6, 16, 5), # n, 16, 10, 10
                          nn.ReLU(),
                          nn.MaxPool2d(2, 2), # n, 16, 5, 5

                          nn.Flatten(), # n, 16 * 5 * 5
                          nn.Linear(16 * 5 * 5, 120),
                          nn.ReLU(),
                          nn.Linear(120, 84),
                          nn.ReLU(),
                          nn.Linear(84, 10))
    return lenet

In [7]:
help(nn.Conv2d)

Help on class Conv2d in module torch.nn.modules.conv:

class Conv2d(_ConvNd)
 |  Conv2d(in_channels: int, out_channels: int, kernel_size: Union[int, tuple[int, int]], stride: Union[int, tuple[int, int]] = 1, padding: Union[str, int, tuple[int, int]] = 0, dilation: Union[int, tuple[int, int]] = 1, groups: int = 1, bias: bool = True, padding_mode: Literal['zeros', 'reflect', 'replicate', 'circular'] = 'zeros', device=None, dtype=None) -> None
 |  
 |  Applies a 2D convolution over an input signal composed of several input
 |  planes.
 |  
 |  In the simplest case, the output value of the layer with input size
 |  :math:`(N, C_{\text{in}}, H, W)` and output :math:`(N, C_{\text{out}}, H_{\text{out}}, W_{\text{out}})`
 |  can be precisely described as:
 |  
 |  .. math::
 |      \text{out}(N_i, C_{\text{out}_j}) = \text{bias}(C_{\text{out}_j}) +
 |      \sum_{k = 0}^{C_{\text{in}} - 1} \text{weight}(C_{\text{out}_j}, k) \star \text{input}(N_i, k)
 |  
 |  
 |  where :math:`\star` is the val

In [6]:
try:
    net = make_lenet()
    X = torch.rand(size=(1, 1, 28, 28), dtype=torch.float32)
    for layer in net:
        X = layer(X)
        print(f'{layer.__class__.__name__:12s} 输出形状: {list(X.shape)}')
except NotImplementedError as e:
    print(f'⚠ {e}')

Conv2d       输出形状: [1, 6, 28, 28]
ReLU         输出形状: [1, 6, 28, 28]
MaxPool2d    输出形状: [1, 6, 14, 14]
Conv2d       输出形状: [1, 16, 10, 10]
ReLU         输出形状: [1, 16, 10, 10]
MaxPool2d    输出形状: [1, 16, 5, 5]
Flatten      输出形状: [1, 400]
Linear       输出形状: [1, 120]
ReLU         输出形状: [1, 120]
Linear       输出形状: [1, 84]
ReLU         输出形状: [1, 84]
Linear       输出形状: [1, 10]


## 题 2 🔧 训练 LeNet

用 Fashion-MNIST 训练 10 个 epoch，观察准确率。

In [8]:
def load_data(batch_size):
    trans = transforms.ToTensor()
    tr = torchvision.datasets.FashionMNIST(root='../data', train=True, transform=trans, download=True)
    te = torchvision.datasets.FashionMNIST(root='../data', train=False, transform=trans, download=True)
    return (data.DataLoader(tr, batch_size, shuffle=True),
            data.DataLoader(te, batch_size, shuffle=False))

def evaluate(net, test_iter):
    net.eval()
    metric = 0.0
    n = 0
    with torch.no_grad():
        for X, y in test_iter:
            metric += float((net(X).argmax(1) == y).sum())
            n += len(X)
    net.train()
    return metric / n

try:
    net = make_lenet()
    loss = nn.CrossEntropyLoss()
    trainer = torch.optim.Adam(net.parameters(), lr=0.01)
    train_iter, test_iter = load_data(256)
    for epoch in range(10):
        for X, y in train_iter:
            l = loss(net(X), y)
            trainer.zero_grad()
            l.backward()
            trainer.step()
        if (epoch + 1) % 5 == 0:
            print(f'epoch {epoch + 1}, 测试准确率 {evaluate(net, test_iter):.3f}')
    print('✓ 训练完成')
except NotImplementedError as e:
    print(f'⚠ {e}')

epoch 5, 测试准确率 0.881
epoch 10, 测试准确率 0.892
✓ 训练完成


## 小结与面试衔接

- LeNet = 卷积提取特征 → 池化下采样 → 全连接分类，是 CNN 的经典范式
- 每层尺寸用 (n−k+2p)/s+1 和池化公式链式推出来
- 一轮 LeNet-5 MNIST 任务（acc>90%）就是这道题的强化版